# 02 岭回归 Ridge Regression

岭回归是在线性回归基础上加了 L2 正则化的模型。它特别适合特征很多、特征彼此相关、普通线性回归系数容易变得很大的情况。


## 0. 学习目标和阅读地图

这一节的重点不是重新学习线性回归，而是理解“正则化”为什么有用。

你应该重点掌握：

1. Ridge 为什么惩罚大系数。
2. `alpha` 如何在“拟合训练数据”和“让模型更稳定”之间做权衡。
3. 为什么使用 Ridge 前通常要标准化特征。
4. Ridge 和 Lasso 的根本区别。


## 1. 数学逻辑

普通线性回归只关心预测误差：

$$\frac{1}{n}\sum_i(y_i-\hat y_i)^2$$

岭回归额外惩罚过大的权重：

$$L(w)=\frac{1}{n}\sum_{i=1}^{n}(y_i-X_iw)^2 + \lambda\sum_{j=1}^{d}w_j^2$$

`lambda` 越大，模型越不愿意使用很大的系数。

直觉：如果很多特征都能解释目标，岭回归会把权重分散得更平滑，而不是让某几个系数特别极端。


## 1.1 推导拆开看：L2 惩罚在做什么

Ridge 的目标函数可以看成两股力量相加：

$$L = \text{预测误差} + \alpha \cdot \text{模型复杂度}$$

其中模型复杂度是：

$$||w||_2^2 = w_1^2 + w_2^2 + \cdots + w_d^2$$

如果某个权重变成 10，它贡献的惩罚是 100；如果两个相关特征各用 5，惩罚是 25 + 25 = 50。因为平方惩罚大权重，Ridge 倾向于把权重分摊得更平滑。

闭式解里的 `alpha I` 也有一个数值稳定作用：

$$w=(X^TX+\alpha I)^{-1}X^Ty$$

当 `X^T X` 接近不可逆时，`alpha I` 会让求解更稳定。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

n = 120
x1 = np.random.normal(size=n)
x2 = x1 + np.random.normal(scale=0.08, size=n)  # 和 x1 高度相关
x3 = np.random.normal(size=n)
X = np.column_stack([x1, x2, x3])
y = 3 * x1 + 3 * x2 + 0.2 * x3 + np.random.normal(scale=1.0, size=n)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 这个例子为什么专门构造相关特征

代码里 `x2 = x1 + 很小噪声`，所以 `x1` 和 `x2` 几乎表达同一件事。普通线性回归可能在这两个特征之间分配出很不稳定的系数。

Ridge 的价值就在这里：它不一定显著降低训练误差，但会让系数更稳、更不容易被采样噪声带偏。


In [ ]:
# 从零实现：岭回归的闭式解
# w = (X^T X + alpha I)^(-1) X^T y

def add_bias(X):
    return np.column_stack([np.ones(len(X)), X])

def ridge_closed_form(X, y, alpha):
    Xb = add_bias(X)
    penalty = np.eye(Xb.shape[1])
    penalty[0, 0] = 0  # 截距项通常不正则化
    return np.linalg.solve(Xb.T @ Xb + alpha * penalty, Xb.T @ y)

for alpha in [0.0, 1.0, 10.0, 100.0]:
    coef = ridge_closed_form(X_train_s, y_train, alpha)
    pred = add_bias(X_test_s) @ coef
    print(f'alpha={alpha:5.1f} | bias={coef[0]: .3f} | weights={np.round(coef[1:], 3)} | MSE={mean_squared_error(y_test, pred):.3f}')


## 1.3 从零实现代码怎么读

闭式解版本做了三件事：

1. `add_bias(X)`：给截距项加一列 1。
2. `penalty = I`：构造 L2 惩罚矩阵。
3. `penalty[0, 0] = 0`：不惩罚截距，因为截距不是特征影响强度。

然后用 `np.linalg.solve` 求解线性方程，比直接求逆更稳定。


In [ ]:
# sklearn 实战：对比普通线性回归和 Ridge
ols = LinearRegression().fit(X_train_s, y_train)
ridge = Ridge(alpha=10.0).fit(X_train_s, y_train)

for name, model in [('LinearRegression', ols), ('Ridge(alpha=10)', ridge)]:
    pred = model.predict(X_test_s)
    print(name)
    print('  weights:', np.round(model.coef_, 3))
    print('  MSE:', round(mean_squared_error(y_test, pred), 3))

alphas = np.logspace(-3, 3, 60)
coefs = []
for a in alphas:
    coefs.append(Ridge(alpha=a).fit(X_train_s, y_train).coef_)
coefs = np.array(coefs)

plt.plot(alphas, coefs)
plt.xscale('log')
plt.title('alpha 越大，Ridge 系数越收缩')
plt.xlabel('alpha')
plt.ylabel('coefficient')
plt.show()


In [ ]:
# 诊断：用交叉验证思路观察 alpha 对测试误差和系数大小的影响
alphas = np.logspace(-3, 3, 30)
test_mse = []
coef_norm = []
for a in alphas:
    m_ridge = Ridge(alpha=a).fit(X_train_s, y_train)
    test_mse.append(mean_squared_error(y_test, m_ridge.predict(X_test_s)))
    coef_norm.append(np.sqrt(np.sum(m_ridge.coef_ ** 2)))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(alphas, test_mse, marker='o')
plt.xscale('log')
plt.title('alpha 与测试 MSE')
plt.xlabel('alpha')
plt.ylabel('MSE')

plt.subplot(1, 2, 2)
plt.plot(alphas, coef_norm, marker='o')
plt.xscale('log')
plt.title('alpha 与权重 L2 范数')
plt.xlabel('alpha')
plt.ylabel('||w||_2')
plt.tight_layout()
plt.show()


## 2.1 如何选择 alpha

`alpha` 太小，Ridge 退化得接近普通线性回归；`alpha` 太大，所有系数都被压得太小，模型会欠拟合。

实战中通常用交叉验证选 `alpha`。观察时同时看两件事：

- 测试误差是否降低或更稳定。
- 系数范数是否随 `alpha` 增大而下降。


## 2. 评价和使用建议

- 回归任务仍然看 `MSE`、`MAE`、`R^2`。
- `alpha` 是关键超参数，通常用交叉验证选择。
- Ridge 不会把系数压成严格的 0；它更像“让所有系数变小”。

## 3. 常见误区

- 使用正则化前通常要标准化特征，否则不同量纲的特征会受到不公平惩罚。
- `alpha` 不是越大越好；太大时会欠拟合。
- Ridge 适合相关特征很多的情况，但不适合直接做特征选择。

## 4. 小实验

- 改 `alpha`，观察系数曲线。
- 增加无关噪声特征，观察 Ridge 和普通线性回归的差别。
- 去掉标准化，看看系数解释是否变得混乱。


## 5. 复习清单

- Ridge 是线性回归加 L2 正则。
- L2 会缩小系数，但通常不会把系数压成精确 0。
- 特征必须先标准化，否则惩罚不公平。
- Ridge 特别适合多重共线性和特征较多的场景。
